# 3) Daily Streak Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Build a daily study habit by recording a 1-line check-in and computing a streak.

## Simple rules

- One line per day (replace allowed).
- Save logs in memory.
- Compute current streak and longest streak.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "03_streak_memory.json"

def today_key() -> str:
    return time.strftime("%Y-%m-%d")

def compute_streak(dates: List[str]) -> int:
    if not dates:
        return 0
    days = sorted(set(dates))
    t = time.strptime(today_key(), "%Y-%m-%d")
    today_epoch_days = int(time.mktime(t) // 86400)

    streak = 0
    for i in range(0, 3650):
        day_str = time.strftime("%Y-%m-%d", time.localtime((today_epoch_days - i) * 86400))
        if day_str in days:
            streak += 1
        else:
            break
    return streak

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action("terminate", {})

    entry = user_text.strip()
    if not entry:
        return Action("notify", {"message": "Write something (e.g., '10 mins arrays')."})

    logs = memory.get("logs", [])
    d = today_key()
    logs = [x for x in logs if x.get("date") != d]
    logs.append({"date": d, "text": entry})
    memory["logs"] = logs

    dates = [x["date"] for x in logs]
    memory["streak"] = compute_streak(dates)
    memory["longest"] = max(int(memory.get("longest", 0)), int(memory["streak"]))

    msg = f"Saved for {d}. Streak: {memory['streak']}. Longest: {memory['longest']}."
    return Action("notify", {"message": msg})

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}

    Tools.notify("Daily Streak Agent is running.")
    Tools.notify("Type your 1-line study log. Type 'stop' to end.")

    user_text = Tools.ask("What did you study today (1 line)?")
    while True:
        action = decide_next_action(user_text, memory)
        out = env.execute(action, memory)
        memory = out["memory"]

        if out["result"] == "terminated":
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved. Bye!")
            break

        env.execute(Action("save_memory", {}), memory)
        user_text = Tools.ask("Update today's line (or stop)")

run_agent()


## Easy extensions

- Save minutes spent.
- Weekly summary.
- Reward messages at milestones.